# Pipeline v2 Visual Assets Lab

Use this notebook to test individual visual scene modules without running the full pipeline. Each visual is driven by a JSON scene spec, rendered through the same Remotion path used by production visuals, and saved under `output_v2/visual_lab/{LAB_RUN_ID}/`.

Workflow:

1. Define scene payloads.
2. Render no-asset code-generated visuals quickly.
3. Optionally resolve screenshots/logos and render asset-backed visuals.
4. Review MP4 outputs and promote good payload patterns back into planner prompts/config.


In [1]:
from __future__ import annotations

from copy import deepcopy
from datetime import datetime, timedelta, timezone
import json
import os
from pathlib import Path
import subprocess
import sys

import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "pipeline_v2").exists():
    raise RuntimeError("Run this notebook from the repository root or output_v2 directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

IST = timezone(timedelta(hours=5, minutes=30))
OUTPUT_V2 = PROJECT_ROOT / "output_v2"
PIPELINE_V2_CONFIG_PATH = PROJECT_ROOT / "src" / "pipeline_v2" / "config" / "workbench.yaml"
REMOTION_ENGINE_DIR = PROJECT_ROOT / "src" / "pipeline_v2" / "visuals" / "remotion_engine"
LAB_RUN_ID = "visual_lab_" + datetime.now(IST).strftime("%Y_%m_%d_%H%M%S")
LAB_ROOT = OUTPUT_V2 / "visual_lab" / LAB_RUN_ID
LAB_INDEX: list[dict] = []

print(f"Project root: {PROJECT_ROOT}")
print(f"Output root:  {OUTPUT_V2}")
print(f"Lab run id:   {LAB_RUN_ID}")
print(f"Lab output:   {LAB_ROOT}")


Project root: /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation
Output root:  /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2
Lab run id:   visual_lab_2026_06_23_135437
Lab output:   /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/visual_lab/visual_lab_2026_06_23_135437


## 1. Helpers

These helpers keep the notebook thin: a scene is a plain dict, and render helpers wrap it in the same input envelopes used by `visuals.render` and `visuals.asset_resolver`.

In [2]:
from src.pipeline_v2.visuals.asset_resolver import run as run_asset_resolver
from src.pipeline_v2.visuals.render import run as run_visual_render


def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(path)
    data = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise ValueError(f"Expected JSON object at {path}")
    return data


def write_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def render_config_templates(value):
    context = {
        "PROJECT_ROOT": str(PROJECT_ROOT),
        "OUTPUT_V2": str(OUTPUT_V2),
        "RUN_ID": LAB_RUN_ID,
    }
    if isinstance(value, str):
        return value.format(**context)
    if isinstance(value, list):
        return [render_config_templates(item) for item in value]
    if isinstance(value, dict):
        return {key: render_config_templates(item) for key, item in value.items()}
    return value


WORKBENCH_CONFIG = yaml.safe_load(PIPELINE_V2_CONFIG_PATH.read_text(encoding="utf-8"))
if not isinstance(WORKBENCH_CONFIG, dict):
    raise ValueError(f"Expected YAML object at {PIPELINE_V2_CONFIG_PATH}")
RUN_PROFILE = WORKBENCH_CONFIG.get("default_run_profile", "daily_48h_all_sources")
PROFILE_CONFIG = deepcopy(WORKBENCH_CONFIG["profiles"][RUN_PROFILE])
VISUALS_CONFIG = deepcopy(PROFILE_CONFIG.get("visuals", {}))


def make_scene(
    scene_id: str,
    render_kind: str,
    payload: dict,
    *,
    duration_seconds: float = 6.0,
    script_excerpt: str = "",
    visual_intent: str = "Visual lab test scene.",
    story_id: str = "visual_lab_story",
    source_record_keys: list[str] | None = None,
    source_urls: list[str] | None = None,
) -> dict:
    return {
        "scene_id": scene_id,
        "scene_order": int(scene_id.split("_")[-1]) if scene_id.split("_")[-1].isdigit() else 1,
        "story_id": story_id,
        "section_type": "story",
        "start_seconds": 0,
        "end_seconds": duration_seconds,
        "duration_seconds": duration_seconds,
        "render_kind": render_kind,
        "script_excerpt": script_excerpt,
        "visual_intent": visual_intent,
        "payload": payload,
        "source_record_keys": source_record_keys or [],
        "source_urls": source_urls or [],
        "covered_unit_orders": [],
    }


def no_asset_manifest(scenes: list[dict], group_name: str) -> dict:
    render_scenes = []
    for index, scene in enumerate(scenes, start=1):
        scene_copy = deepcopy(scene)
        scene_copy["scene_order"] = index
        scene_copy["resolved_asset"] = {
            "scene_id": scene_copy["scene_id"],
            "scene_order": index,
            "story_id": scene_copy.get("story_id", ""),
            "render_kind": scene_copy["render_kind"],
            "asset_type": "none",
            "status": "ready",
            "asset_path": "",
            "error": "",
        }
        render_scenes.append(scene_copy)
    return {
        "package_type": "visual_asset_manifest",
        "run_id": LAB_RUN_ID,
        "generated_at": datetime.now(IST).isoformat(),
        "output_dir": str(LAB_ROOT / group_name),
        "visual_scene_plan": {
            "package_type": "visual_scene_plan",
            "scene_policy": {"max_scene_seconds": 14},
            "scene_timeline": scenes,
        },
        "resolved_assets": [scene["resolved_asset"] for scene in render_scenes],
        "render_scenes": render_scenes,
    }


def render_scene_group(scenes: list[dict], group_name: str, *, quality_preset: str = "1080p") -> dict:
    output_dir = LAB_ROOT / group_name / "render"
    render_config = render_config_templates(deepcopy(VISUALS_CONFIG.get("render", {})))
    render_config.update(
        {
            "output_dir": str(output_dir),
            "force": True,
            "quality_preset": quality_preset,
            "remotion_engine_dir": str(REMOTION_ENGINE_DIR),
            "render_timeout_seconds": max(300, int(render_config.get("render_timeout_seconds", 240))),
        }
    )
    render_input = {
        "run": {"run_id": LAB_RUN_ID, "generated_at": datetime.now(IST).isoformat()},
        "config": render_config,
        "input": {
            "visual_asset_resolution": {
                "status": "ok",
                "data": {
                    "run_id": LAB_RUN_ID,
                    "generated_at": datetime.now(IST).isoformat(),
                    "visual_asset_manifest": no_asset_manifest(scenes, group_name),
                },
                "summary": {},
                "handoff": {"ready_for_visual_render": True},
            }
        },
    }
    write_json(LAB_ROOT / group_name / "render_input.json", render_input)
    result = run_visual_render(render_input)
    summary = result["summary"]
    LAB_INDEX.append(
        {
            "group_name": group_name,
            "mode": "no_asset_render",
            "status": result["status"],
            "scene_count": summary["requested_scene_count"],
            "rendered_scene_count": summary["rendered_scene_count"],
            "failed_scene_count": summary["failed_scene_count"],
            "output_dir": str(output_dir),
            "summary": summary,
        }
    )
    write_json(LAB_ROOT / "visual_lab_index.json", LAB_INDEX)
    print(f"{group_name}: status={result['status']} rendered={summary['rendered_scene_count']}/{summary['requested_scene_count']}")
    print(f"Output: {output_dir}")
    return result


def resolve_and_render_scene_group(scenes: list[dict], group_name: str, *, quality_preset: str = "1080p") -> dict:
    asset_output_dir = LAB_ROOT / group_name / "asset_resolution"
    render_output_dir = LAB_ROOT / group_name / "render"
    asset_config = render_config_templates(deepcopy(VISUALS_CONFIG.get("asset_resolver", {})))
    asset_config.update({"output_dir": str(asset_output_dir), "force": True, "target_width": 1920, "target_height": 1080})
    if not asset_config.get("logos", {}).get("local_logo_dir"):
        asset_config.setdefault("logos", {})["local_logo_dir"] = str(REMOTION_ENGINE_DIR / "assets" / "logos")
    visual_scene_plan = {
        "package_type": "visual_scene_plan",
        "selected_format": "visual_lab",
        "scene_policy": {"max_scene_seconds": 14},
        "scene_timeline": scenes,
    }
    asset_input = {
        "run": {"run_id": LAB_RUN_ID, "generated_at": datetime.now(IST).isoformat()},
        "config": asset_config,
        "input": {
            "visual_scene_planning": {
                "status": "ok",
                "data": {
                    "run_id": LAB_RUN_ID,
                    "generated_at": datetime.now(IST).isoformat(),
                    "selected_format": "visual_lab",
                    "visual_scene_plan": visual_scene_plan,
                },
                "summary": {},
                "handoff": {"ready_for_visual_asset_resolution": True},
            }
        },
    }
    write_json(LAB_ROOT / group_name / "asset_resolution_input.json", asset_input)
    asset_result = run_asset_resolver(asset_input)
    if not asset_result.get("handoff", {}).get("ready_for_visual_render", False):
        raise RuntimeError(asset_result.get("summary", {}))
    render_config = render_config_templates(deepcopy(VISUALS_CONFIG.get("render", {})))
    render_config.update(
        {
            "output_dir": str(render_output_dir),
            "force": True,
            "quality_preset": quality_preset,
            "remotion_engine_dir": str(REMOTION_ENGINE_DIR),
            "render_timeout_seconds": max(300, int(render_config.get("render_timeout_seconds", 240))),
        }
    )
    render_input = {
        "run": {"run_id": LAB_RUN_ID, "generated_at": datetime.now(IST).isoformat()},
        "config": render_config,
        "input": {"visual_asset_resolution": asset_result},
    }
    write_json(LAB_ROOT / group_name / "render_input.json", render_input)
    render_result = run_visual_render(render_input)
    summary = render_result["summary"]
    LAB_INDEX.append(
        {
            "group_name": group_name,
            "mode": "asset_resolve_then_render",
            "asset_status": asset_result["status"],
            "render_status": render_result["status"],
            "scene_count": summary["requested_scene_count"],
            "rendered_scene_count": summary["rendered_scene_count"],
            "failed_scene_count": summary["failed_scene_count"],
            "asset_output_dir": str(asset_output_dir),
            "render_output_dir": str(render_output_dir),
            "asset_summary": asset_result.get("summary", {}),
            "render_summary": summary,
        }
    )
    write_json(LAB_ROOT / "visual_lab_index.json", LAB_INDEX)
    print(f"{group_name}: asset={asset_result['status']} render={render_result['status']} rendered={summary['rendered_scene_count']}/{summary['requested_scene_count']}")
    print(f"Output: {render_output_dir}")
    return render_result


## 2. Sample No-Asset Scene Gallery

This renders one scene for each code-generated visual family. These scenes do not need screenshots or logos, so they are the fastest way to iterate on motion design.

In [3]:
no_asset_gallery = [
    make_scene(
        "scene_001",
        "kinetic_type_scene",
        {
            "phrases": ["AI leaves the chat box", "and enters the workflow"],
            "emphasis_words": ["AI", "workflow"],
            "animation_style": "word_snap",
            "background_motif": "abstract_ai",
        },
        duration_seconds=5,
        script_excerpt="AI is moving out of the chat box and into everyday work.",
    ),
    make_scene(
        "scene_002",
        "bullet_build_scene",
        {
            "headline": "Why this matters",
            "points": ["New workflows", "Enterprise pressure", "Developer adoption"],
            "emphasis_words": ["workflows", "pressure", "adoption"],
            "build_style": "stack_reveal",
            "background_motif": "newsroom_grid",
        },
        duration_seconds=7,
    ),
    make_scene(
        "scene_003",
        "data_counter_scene",
        {
            "headline": "The scale signal",
            "metrics": [
                {"label": "Funding", "value": "$12B", "unit": "", "context": "startup capital at unprecedented scale"},
                {"label": "Valuation", "value": "$41B", "unit": "", "context": "before a public product exists"},
            ],
            "emphasis_label": "capital is becoming the product signal",
            "motion_style": "count_up",
            "background_motif": "device_glow",
        },
        duration_seconds=7,
    ),
    make_scene(
        "scene_004",
        "trend_line_scene",
        {
            "headline": "Pressure keeps rising",
            "trend_direction": "pressure_building",
            "y_axis_label": "Market pressure",
            "x_axis_label": "This week",
            "annotations": ["cloud demand", "GPU race", "enterprise AI"],
            "motion_style": "draw_line",
            "background_motif": "abstract_ai",
        },
        duration_seconds=7,
    ),
    make_scene(
        "scene_005",
        "ecosystem_map_scene",
        {
            "headline": "Who connects to whom",
            "nodes": [
                {"id": "openai", "label": "OpenAI", "role": "models"},
                {"id": "aws", "label": "AWS", "role": "cloud"},
                {"id": "nvidia", "label": "NVIDIA", "role": "compute"},
                {"id": "enterprise", "label": "Enterprise teams", "role": "users"},
            ],
            "edges": [
                {"source": "openai", "target": "enterprise", "label": "AI tools"},
                {"source": "aws", "target": "enterprise", "label": "deployment"},
                {"source": "nvidia", "target": "aws", "label": "compute"},
            ],
            "focus_entity": "Enterprise teams",
            "motion_style": "pulse_connections",
            "background_motif": "device_glow",
        },
        duration_seconds=8,
    ),
    make_scene(
        "scene_006",
        "architecture_flow_scene",
        {
            "headline": "From request to result",
            "steps": [
                {"label": "User task", "detail": "plain language"},
                {"label": "Agent", "detail": "plans actions"},
                {"label": "Tools", "detail": "calls APIs"},
                {"label": "Outcome", "detail": "finished work"},
            ],
            "flow_direction": "left_to_right",
            "active_step_label": "Outcome",
            "motion_style": "packet_flow",
            "background_motif": "newsroom_grid",
        },
        duration_seconds=8,
    ),
    make_scene(
        "scene_007",
        "entity_compare_scene",
        {
            "headline": "Platform battle lines",
            "entities": ["OpenAI", "Anthropic", "Google", "Meta"],
            "comparison_axis": "Enterprise AI control",
            "entity_notes": ["developer tools", "safety controls", "search + apps", "open models"],
            "motion_style": "matrix",
        },
        duration_seconds=7,
    ),
    make_scene(
        "scene_008",
        "recap_sequence_scene",
        {
            "headline": "The bigger shift",
            "recap_items": ["AI infrastructure", "agent workflows", "enterprise adoption", "physical-world systems"],
            "final_takeaway": "AI is becoming operating infrastructure.",
            "motion_style": "story_rhythm",
            "background_motif": "abstract_ai",
        },
        duration_seconds=8,
    ),
    make_scene(
        "scene_009",
        "transition_stinger_scene",
        {
            "transition_label": "Next signal",
            "subtitle": "the race moves from demos to deployment",
            "motion_style": "signal_wipe",
            "background_motif": "abstract_ai",
        },
        duration_seconds=4,
    ),
    make_scene(
        "scene_010",
        "bar_chart_scene",
        {
            "headline": "Where the budget goes",
            "bars": [
                {"label": "Compute", "value": "82", "display_value": "82%", "context": "GPU demand"},
                {"label": "Data", "value": "54", "display_value": "54%", "context": "training + evals"},
                {"label": "Product", "value": "38", "display_value": "38%", "context": "workflow polish"},
                {"label": "Support", "value": "24", "display_value": "24%", "context": "enterprise rollout"},
            ],
            "unit": "%",
            "emphasis_label": "comparison should feel instant",
            "motion_style": "grow_bars",
            "background_motif": "newsroom_grid",
        },
        duration_seconds=7,
    ),
    make_scene(
        "scene_011",
        "bar_race_scene",
        {
            "headline": "The deployment race tightens",
            "items": [
                {"label": "Cloud", "start_value": "65", "end_value": "84", "display_value": "84"},
                {"label": "On-device", "start_value": "24", "end_value": "76", "display_value": "76"},
                {"label": "Open models", "start_value": "31", "end_value": "69", "display_value": "69"},
                {"label": "Agent apps", "start_value": "18", "end_value": "72", "display_value": "72"},
            ],
            "leader_label": "the race is no longer one lane",
            "motion_style": "rank_shift",
            "background_motif": "device_glow",
        },
        duration_seconds=7,
    ),
    make_scene(
        "scene_012",
        "ranked_list_scene",
        {
            "headline": "Three signals to watch",
            "items": [
                {"rank": 1, "label": "Workflow fit", "value": "high", "detail": "does it remove real steps?"},
                {"rank": 2, "label": "Distribution", "value": "wide", "detail": "does it ship inside tools?"},
                {"rank": 3, "label": "Trust", "value": "fragile", "detail": "can teams verify output?"},
            ],
            "emphasis_label": "rank the decision, not the hype",
            "motion_style": "staggered_rank",
            "background_motif": "abstract_ai",
        },
        duration_seconds=7,
    ),
    make_scene(
        "scene_013",
        "benchmark_delta_scene",
        {
            "headline": "The delta is the story",
            "benchmarks": [
                {"label": "Latency", "before": "1.0x", "after": "2.4x", "delta": "+140%", "context": "faster response loop"},
                {"label": "Cost", "before": "$1.00", "after": "$0.62", "delta": "-38%", "context": "cheaper deployment"},
                {"label": "Context", "before": "128K", "after": "1M", "delta": "7.8x", "context": "larger workflow memory"},
            ],
            "winner_label": "delta beats headline hype",
            "motion_style": "delta_snap",
            "background_motif": "device_glow",
        },
        duration_seconds=7,
    ),
    make_scene(
        "scene_014",
        "process_zoom_scene",
        {
            "headline": "Zoom into the agent loop",
            "steps": [
                {"label": "User request", "detail": "intent arrives"},
                {"label": "Planner", "detail": "breaks work into steps"},
                {"label": "Tool call", "detail": "acts in the real app"},
                {"label": "Verification", "detail": "checks what changed"},
                {"label": "Answer", "detail": "returns useful output"},
            ],
            "focus_step_label": "Tool call",
            "zoom_note": "This is where an assistant stops being a chatbot and starts changing the workflow.",
            "motion_style": "focus_zoom",
            "background_motif": "newsroom_grid",
        },
        duration_seconds=8,
    ),
    make_scene(
        "scene_015",
        "browser_walkthrough_scene",
        {
            "headline": "Watch the source come alive",
            "url": "https://example.com/blog/agent-workflows",
            "browser_title": "AI moves from chat into real tools",
            "focus_text": "AI moves from chat into real tools and workflows.",
            "walkthrough_steps": [
                {"label": "Open", "detail": "Start at the product announcement"},
                {"label": "Zoom", "detail": "Highlight the workflow claim"},
                {"label": "Connect", "detail": "Tie the source to the story angle"},
            ],
            "highlight_phrases": ["agent workflows", "real tools", "enterprise rollout"],
            "background_motif": "newsroom_grid",
        },
        duration_seconds=6,
    ),
    make_scene(
        "scene_016",
        "decision_tree_scene",
        {
            "headline": "The agent has to choose a path",
            "root_label": "User request arrives",
            "selected_path_label": "Call a tool",
            "branches": [
                {"label": "Answer from memory", "outcome": "fast but risky"},
                {"label": "Call a tool", "outcome": "grounded action"},
                {"label": "Ask approval", "outcome": "human checkpoint"},
                {"label": "Hand off", "outcome": "specialist workflow"},
            ],
            "background_motif": "abstract_ai",
        },
        duration_seconds=6,
    ),
    make_scene(
        "scene_017",
        "capability_matrix_scene",
        {
            "headline": "Pick the model by the job",
            "columns": ["Speed", "Cost", "Context", "Tools"],
            "highlight_cell": "tool use",
            "rows": [
                {"label": "Mini model", "values": ["fast", "low", "short", "basic"], "highlight": "Speed"},
                {"label": "Flagship", "values": ["steady", "medium", "long", "tool use"], "highlight": "tool use"},
                {"label": "Reasoning", "values": ["slower", "higher", "deep", "planning"], "highlight": "planning"},
            ],
            "background_motif": "device_glow",
        },
        duration_seconds=6,
    ),
    make_scene(
        "scene_018",
        "timeline_zoom_scene",
        {
            "headline": "A quiet signal turns into a market move",
            "focus_event_label": "Launch window",
            "events": [
                {"time": "Monday", "label": "Rumor appears", "detail": "The first source hints at the shift"},
                {"time": "Wednesday", "label": "Docs update", "detail": "Developers notice new references"},
                {"time": "Friday", "label": "Launch window", "detail": "The public announcement changes the frame"},
                {"time": "Next", "label": "Competitors react", "detail": "The ecosystem starts moving"},
            ],
            "background_motif": "newsroom_grid",
        },
        duration_seconds=6,
    ),
    make_scene(
        "scene_019",
        "document_stack_scene",
        {
            "headline": "The story is backed by a stack of evidence",
            "stack_summary": "Releases, docs, and developer chatter point the same way.",
            "documents": [
                {"title": "Product announcement", "source": "Company blog", "document_type": "release"},
                {"title": "API docs update", "source": "Developer portal", "document_type": "docs"},
                {"title": "GitHub issue spike", "source": "Open source repo", "document_type": "repo"},
                {"title": "Analyst note", "source": "Market coverage", "document_type": "report"},
            ],
            "background_motif": "newsroom_grid",
        },
        duration_seconds=6,
    ),
    make_scene(
        "scene_020",
        "signal_radar_scene",
        {
            "headline": "Three signals say this trend is heating up",
            "scan_label": "agentic workflows",
            "signals": [
                {"label": "Developer adoption", "strength": 82, "detail": "tooling mentions rise"},
                {"label": "Cloud integration", "strength": 74, "detail": "more platform hooks"},
                {"label": "Enterprise pressure", "strength": 67, "detail": "buyers ask for control"},
                {"label": "Model cost", "strength": 58, "detail": "cheaper routing matters"},
            ],
            "background_motif": "abstract_ai",
        },
        duration_seconds=6,
    ),
    make_scene(
        "scene_021",
        "interface_transform_scene",
        {
            "headline": "The interface shifts from chat to action",
            "transformation_label": "prompt -> plan -> tool result",
            "before": {"title": "Before", "elements": ["Ask a chatbot", "Copy the answer", "Do the work manually"]},
            "after": {"title": "After", "elements": ["Describe the goal", "Agent plans steps", "Tools execute the workflow"]},
            "background_motif": "device_glow",
        },
        duration_seconds=6,
    ),
]

no_asset_result = render_scene_group(no_asset_gallery, "01_no_asset_gallery", quality_preset="1080p")


01_no_asset_gallery: status=ok rendered=9/9
Output: /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/visual_lab/visual_lab_2026_06_23_135437/01_no_asset_gallery/render


## 3. Variant Tests

Use this section to test how the same visual module behaves with different payload shapes. Add/edit payloads here while designing better scene modules.

In [4]:
data_and_transition_variants = [
    make_scene(
        "scene_001",
        "data_counter_scene",
        {
            "headline": "Model scale jump",
            "metrics": [
                {"label": "Context", "value": "1M", "unit": "tokens", "context": "longer workflows fit in memory"},
                {"label": "Latency", "value": "2x", "unit": "faster", "context": "fewer waits for operators"},
                {"label": "Cost", "value": "-40%", "unit": "", "context": "deployment gets easier"},
            ],
            "emphasis_label": "numbers should feel concrete",
            "motion_style": "scale_bars",
            "background_motif": "newsroom_grid",
        },
        duration_seconds=7,
    ),
    make_scene(
        "scene_002",
        "trend_line_scene",
        {
            "headline": "Adoption bends upward",
            "trend_direction": "flat_then_jump",
            "y_axis_label": "Adoption",
            "x_axis_label": "Launch window",
            "annotations": ["pilot", "rollout", "workflow shift"],
            "motion_style": "snap_breakout",
            "background_motif": "device_glow",
        },
        duration_seconds=7,
    ),
    make_scene(
        "scene_003",
        "transition_stinger_scene",
        {
            "transition_label": "Cloud to compute",
            "subtitle": "watch the infrastructure layer",
            "motion_style": "radar_ping",
            "background_motif": "device_glow",
        },
        duration_seconds=4,
    ),
    make_scene(
        "scene_004",
        "transition_stinger_scene",
        {
            "transition_label": "Safety switch",
            "subtitle": "when policy becomes product behavior",
            "motion_style": "glitch_snap",
            "background_motif": "abstract_ai",
        },
        duration_seconds=4,
    ),
]

variant_result = render_scene_group(data_and_transition_variants, "02_data_trend_transition_variants", quality_preset="1080p")


02_data_trend_transition_variants: status=ok rendered=4/4
Output: /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/visual_lab/visual_lab_2026_06_23_135437/02_data_trend_transition_variants/render


## 4. Optional Asset-Backed Visual Tests

This cell tests screenshot/logo assets. It is slower because it opens Playwright and may hit live websites. Set `RUN_ASSET_BACKED_TESTS = False` if you want a render-only quick pass.

In [5]:
RUN_ASSET_BACKED_TESTS = True

fixture_dir = LAB_ROOT / "fixtures"
fixture_dir.mkdir(parents=True, exist_ok=True)
visual_lab_source_fixture = fixture_dir / "source_article_fixture.html"
visual_lab_source_fixture.write_text(
    """
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <title>Visual Lab Source Fixture</title>
  <style>
    body { margin: 0; background: #f7fafc; color: #102033; font-family: Avenir Next, Helvetica, Arial, sans-serif; }
    main { max-width: 980px; margin: 0 auto; padding: 86px 72px 120px; }
    .eyebrow { color: #1272aa; font-size: 18px; font-weight: 900; letter-spacing: .14em; text-transform: uppercase; }
    h1 { font-size: 68px; line-height: .95; letter-spacing: -.055em; margin: 18px 0 24px; }
    .dek { font-size: 28px; line-height: 1.22; color: #496276; font-weight: 680; max-width: 820px; }
    .callout { margin-top: 54px; padding: 34px 38px; border-radius: 28px; background: #e8f6ff; border: 2px solid #8bdcff; }
    .callout strong { display: block; font-size: 34px; line-height: 1.04; letter-spacing: -.035em; }
    .callout p { font-size: 24px; line-height: 1.28; color: #315067; margin: 14px 0 0; }
    article p { font-size: 24px; line-height: 1.42; color: #263c4f; }
  </style>
</head>
<body>
  <main>
    <article>
      <div class="eyebrow">Visual Lab Fixture</div>
      <h1>AI moves from demos into daily workflows</h1>
      <p class="dek">This fixture gives the screenshot renderer a stable article-like page, so visual tests are about composition rather than website randomness.</p>
      <div class="callout">
        <strong>Teams are starting to treat AI agents as workflow infrastructure.</strong>
        <p>The important visual beat is not just a headline. It is the shift from a chatbot interface into tools, documents, and operating processes.</p>
      </div>
      <p>For a YouTube explainer, this kind of source view should feel like evidence. The camera can push toward a headline, then highlight the claim that connects to the narration.</p>
    </article>
  </main>
</body>
</html>
    """.strip(),
    encoding="utf-8",
)
fixture_url = visual_lab_source_fixture.resolve().as_uri()

asset_backed_scenes = [
    make_scene(
        "scene_001",
        "source_article_scene",
        {
            "url": fixture_url,
            "source_record_key": "visual_lab_source_fixture",
            "source_label": "local source fixture",
            "target_text": "Teams are starting to treat AI agents as workflow infrastructure.",
            "highlight_phrases": ["AI agents", "workflow infrastructure"],
            "crop_strategy": "headline_then_body",
            "overlay_headline": "AI moves into daily workflows",
            "motion_style": "slow_push",
        },
        duration_seconds=7,
        source_record_keys=["visual_lab_source_fixture"],
        source_urls=[fixture_url],
    ),
    make_scene(
        "scene_002",
        "quote_or_claim_highlight_scene",
        {
            "url": fixture_url,
            "source_record_key": "visual_lab_source_fixture",
            "claim_text": "AI agents are becoming workflow infrastructure.",
            "supporting_text": "The article fixture connects source evidence to narration beats.",
            "highlight_phrases": ["AI agents", "workflow infrastructure", "narration"],
            "overlay_headline": "Source-backed claim",
            "motion_style": "highlight_scan",
        },
        duration_seconds=7,
        source_record_keys=["visual_lab_source_fixture"],
        source_urls=[fixture_url],
    ),
    make_scene(
        "scene_003",
        "logo_reveal_scene",
        {
            "entities": ["OpenAI", "Developers", "Enterprise teams"],
            "primary_entity": "OpenAI",
            "logo_strategy": "simple_icons_then_favicon_then_text",
            "companion_text": "AI moves into workplace training",
            "motion_style": "spotlight",
        },
        duration_seconds=6,
    ),
]

if RUN_ASSET_BACKED_TESTS:
    asset_result = resolve_and_render_scene_group(asset_backed_scenes, "03_asset_backed_gallery", quality_preset="1080p")
else:
    print("Asset-backed tests are skipped. Set RUN_ASSET_BACKED_TESTS = True and rerun this cell to render them.")


Asset-backed tests are skipped. Set RUN_ASSET_BACKED_TESTS = True and rerun this cell to render them.


## 5. Lab Index

This writes a small markdown index with output paths so we can compare visual variants without hunting through folders.

In [6]:
def write_lab_markdown_index() -> Path:
    lines = [
        f"# Visual Lab Index - {LAB_RUN_ID}",
        "",
        f"Generated: {datetime.now(IST).isoformat()}",
        f"Project root: `{PROJECT_ROOT}`",
        "",
    ]
    for item in LAB_INDEX:
        lines.extend(
            [
                f"## {item['group_name']}",
                "",
                f"- Mode: `{item['mode']}`",
                f"- Status: `{item.get('status', item.get('render_status', ''))}`",
                f"- Rendered: `{item['rendered_scene_count']}/{item['scene_count']}`",
                f"- Failed: `{item['failed_scene_count']}`",
                f"- Output: `{item.get('output_dir', item.get('render_output_dir', ''))}`",
                "",
            ]
        )
    path = LAB_ROOT / "visual_lab_index.md"
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(lines), encoding="utf-8")
    write_json(LAB_ROOT / "visual_lab_index.json", LAB_INDEX)
    return path


index_path = write_lab_markdown_index()
print(f"Lab root:   {LAB_ROOT}")
print(f"Index JSON: {LAB_ROOT / 'visual_lab_index.json'}")
print(f"Index MD:   {index_path}")
LAB_INDEX


Lab root:   /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/visual_lab/visual_lab_2026_06_23_135437
Index JSON: /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/visual_lab/visual_lab_2026_06_23_135437/visual_lab_index.json
Index MD:   /Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/visual_lab/visual_lab_2026_06_23_135437/visual_lab_index.md


[{'group_name': '01_no_asset_gallery',
  'mode': 'no_asset_render',
  'status': 'ok',
  'scene_count': 9,
  'rendered_scene_count': 9,
  'failed_scene_count': 0,
  'output_dir': '/Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/visual_lab/visual_lab_2026_06_23_135437/01_no_asset_gallery/render',
  'summary': {'requested_scene_count': 9,
   'rendered_scene_count': 9,
   'failed_scene_count': 0,
   'output_dir': '/Users/manikantamb/_MyFiles/Work_/MyWork_Local/AI_Content_Creation/output_v2/visual_lab/visual_lab_2026_06_23_135437/01_no_asset_gallery/render',
   'failure_reason': '',
   'warning_count': 0,
   'render_kind_counts': {'kinetic_type_scene': 1,
    'bullet_build_scene': 1,
    'data_counter_scene': 1,
    'trend_line_scene': 1,
    'ecosystem_map_scene': 1,
    'architecture_flow_scene': 1,
    'entity_compare_scene': 1,
    'recap_sequence_scene': 1,
    'transition_stinger_scene': 1},
   'status_counts': {'rendered': 9},
   'render_settings': {'qual